| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 3: Data Preparation | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 01-12-2025 | Júlio César e Lays de Freitas | Rascunho |

Esse Notebook contém o **pré-processamento dos dados**.

### BIBLIOTECAS

In [39]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from datetime import date

### CARREGAMENTO DOS DADOS

In [40]:
# Listar os arquivos CSV na pasta 'uploads'
arquivos_csv = glob.glob('uploads/processos_*.csv')

# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivos_csv:
    
    df_temp = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    dfs.append(df_temp)

dataset = pd.concat(dfs, ignore_index=True)

print("\n=== Arquivo carregado com sucesso! ===")
print("Dimensões (linhas, colunas):", dataset.shape)


=== Arquivo carregado com sucesso! ===
Dimensões (linhas, colunas): (3245632, 10)


### GRAVANDO UMA CÓPIA PARA TRABALHO

In [41]:
df = dataset.copy()

### AMOSTRA DOS DADOS

In [42]:
df.head()

,processo,data_distribuicao,data_baixa,entrancia,comarca,serventia,nome_area_acao,is_segredo_justica,codg_classe,codg_assuntos
0,0119071.75.2004.8.09.0051,2022-05-25,2022-06-30,FINAL,GOIÂNIA,2ª Vara Cível,upj civel,False,7.0,10671
1,0168391.94.2004.8.09.0051,2022-05-20,2022-05-20,FINAL,GOIÂNIA,3ª Vara Cível,upj civel,False,7.0,10671
2,0189657.40.2004.8.09.0051,2022-06-02,2024-01-22,FINAL,GOIÂNIA,31ª Vara Cível,upj civel,False,7.0,10671
3,0197944.89.2004.8.09.0051,2022-06-07,2022-10-07,FINAL,GOIÂNIA,22ª Vara Cível,upj civel,False,7.0,10671
4,0211274.56.2004.8.09.0051,2022-06-09,2022-08-03,FINAL,GOIÂNIA,8ª Vara Cível,upj civel,False,7.0,10671


### LIMPEZA E TRATAMENTO DOS DADOS

In [43]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# Converter colunas de data para datetime com tratamento de erros
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

### CONSTRUÇÃO DO DATAFRAME DE TREINO E TESTE

In [44]:
# CRIAÇÃO DAS ESTATÍSTICAS POR MÊS ('comarca' e 'serventia') >> Revisado
# --- 1. PREPARAÇÃO DOS DADOS ---
# Extração de componentes de data
print("Extraindo datas...")
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['mes_distribuicao'] = df['data_distribuicao'].dt.month
df['dia_distribuicao'] = df['data_distribuicao'].dt.day

df['ano_baixa'] = df['data_baixa'].dt.year
df['mes_baixa'] = df['data_baixa'].dt.month
df['dia_baixa'] = df['data_baixa'].dt.day

# Chaves de agrupamento
grouping_keys = ['comarca', 'serventia']

# ==============================================================================
# FUNÇÃO GENÉRICA DE CÁLCULO (Para evitar repetição de código)
# ==============================================================================
def calcular_estatisticas_cohort(df_main, cols_dist, cols_baixa, nome_periodo):
    """
    Calcula Distribuídos, Baixados e Pendentes.
    REVISÃO DOS CÁLCULOS:
      - Distribuídos: contagem por data_distribuicao (entrada)
      - Baixados: contagem por data_baixa (referência), dentro do par entrada->referência
      - Pendentes: contagem de data_baixa nula/vazia (NaT), agrupada por entrada
    """
    
    # 1. Calcular TOTAL DE DISTRIBUÍDOS
    cols_group_dist = cols_dist + grouping_keys
    df_dist = df_main.groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Distribuídos{nome_periodo}')
    
    # 2. Calcular TOTAL DE BAIXADOS (somente registros com baixa)
    cols_group_baixa = cols_dist + cols_baixa + grouping_keys
    df_baixa = df_main.dropna(subset=cols_baixa).groupby(cols_group_baixa)['processo'].nunique().reset_index(name=f'Baixados{nome_periodo}')
    
    # 3. Calcular TOTAL DE PENDENTES (data_baixa nula/vazia -> componentes de baixa NaN)
    df_pend = df_main[df_main[cols_baixa[0]].isna()].groupby(cols_group_dist)['processo'].nunique().reset_index(name=f'Pendentes{nome_periodo}')
    
    # 4. CRIAÇÃO DO GRID (Cross Join)
    unique_dist = df_dist[cols_dist].drop_duplicates()
    unique_baixa = df_main[cols_baixa].dropna().drop_duplicates()
    unique_units = df_main[grouping_keys].drop_duplicates()
    
    # Cross Join 1: Datas de Dist x Datas de Baixa (usando merge dummy para performance)
    df_dates = pd.merge(
        unique_dist.assign(key=1), 
        unique_baixa.assign(key=1), 
        on='key'
    ).drop('key', axis=1)
    
    # --- FILTRO DE DATAS ---
    if len(cols_dist) == 1: # Anual
        df_dates = df_dates[df_dates[cols_baixa[0]] >= df_dates[cols_dist[0]]]
        
    elif len(cols_dist) == 2: # Mensal
        ano_d = df_dates[cols_dist[0]].astype(int).astype(str)
        mes_d = df_dates[cols_dist[1]].astype(int).astype(str)
        
        ano_b = df_dates[cols_baixa[0]].astype(int).astype(str)
        mes_b = df_dates[cols_baixa[1]].astype(int).astype(str)
        
        d_dist = pd.to_datetime(ano_d + '-' + mes_d + '-01')
        d_baixa = pd.to_datetime(ano_b + '-' + mes_b + '-01')
        
        df_dates = df_dates[d_baixa >= d_dist]

    # Cross Join 2: (Datas) x (Comarca/Serventia)
    df_grid = pd.merge(
        df_dates.assign(key=1),
        unique_units.assign(key=1),
        on='key'
    ).drop('key', axis=1)
    
    # 5. MERGES (Juntar dados reais no Grid)
    df_final = pd.merge(df_grid, df_dist, on=cols_dist + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_baixa, on=cols_dist + cols_baixa + grouping_keys, how='left')
    df_final = pd.merge(df_final, df_pend, on=cols_dist + grouping_keys, how='left')
    
    # Preencher Zeros
    df_final[f'Distribuídos{nome_periodo}'] = df_final[f'Distribuídos{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Baixados{nome_periodo}'] = df_final[f'Baixados{nome_periodo}'].fillna(0).astype(int)
    df_final[f'Pendentes{nome_periodo}'] = df_final[f'Pendentes{nome_periodo}'].fillna(0).astype(int)
    
    # Filtrar apenas onde houve distribuição
    df_final = df_final[df_final[f'Distribuídos{nome_periodo}'] > 0].copy()
    
    # 6. TAXA DE CONGESTIONAMENTO (com a definição solicitada)
    soma = df_final[f'Baixados{nome_periodo}'] + df_final[f'Pendentes{nome_periodo}']
    df_final[f'Taxa de Congestionamento{nome_periodo} (%)'] = np.where(
        soma > 0, (df_final[f'Pendentes{nome_periodo}'] / soma) * 100, 0
    ).round(2)

    # 7. CONVERSÃO FINAL PARA INTEIRO (NOVO BLOCO)
    cols_tempo = cols_dist + cols_baixa
    for col in cols_tempo:
        if col in df_final.columns:
            df_final[col] = df_final[col].astype(int)

    return df_final

# ==============================================================================
# 2. CÁLCULOS MENSAIS
# ==============================================================================
print("Calculando estatísticas mensais...")
df_estatisticas_mes = calcular_estatisticas_cohort(
    df, 
    cols_dist=['ano_distribuicao', 'mes_distribuicao'], 
    cols_baixa=['ano_baixa', 'mes_baixa'], 
    nome_periodo='_mes'
)

# Ajustes finais de colunas e ordenação
df_estatisticas_mes = df_estatisticas_mes.rename(columns={'ano_distribuicao': 'ano', 'mes_distribuicao': 'mes'})
cols_order_mes = ['ano', 'mes', 'ano_baixa', 'mes_baixa', 'comarca', 'serventia', 
                  'Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes', 'Taxa de Congestionamento_mes (%)']

df_estatisticas_mes = df_estatisticas_mes[cols_order_mes].sort_values(
    by=['ano', 'mes', 'ano_baixa', 'mes_baixa', 'comarca', 'serventia'],
    ascending=[False, False, False, False, True, True]
)

df_estatisticas_mes = df_estatisticas_mes.rename(columns={
    'ano_baixa': 'ano_baixa', 
    'mes_baixa': 'mes_baixa',
    'ano': 'ano_distribuicao',
    'mes': 'mes_distribuicao'   
})

# ==============================================================================
# 3. CÁLCULOS ANUAIS
# ==============================================================================
print("Calculando estatísticas anuais...")
df_estatisticas_anuais = calcular_estatisticas_cohort(
    df, 
    cols_dist=['ano_distribuicao'], 
    cols_baixa=['ano_baixa'], 
    nome_periodo=''
)

# Ajustes finais de colunas e ordenação
df_estatisticas_anuais = df_estatisticas_anuais.rename(columns={'ano_distribuicao': 'ano'})
cols_order = ['ano', 'ano_baixa', 'comarca', 'serventia', 'Distribuídos', 'Baixados', 'Pendentes', 'Taxa de Congestionamento (%)']
df_estatisticas_anuais = df_estatisticas_anuais[cols_order].sort_values(
    by=['ano', 'ano_baixa', 'comarca', 'serventia'], 
    ascending=[False, False, True, True]
)

df_estatisticas_anuais = df_estatisticas_anuais.rename(columns={
    'ano_baixa': 'ano_baixa', 
    'ano': 'ano_distribuicao'
})

print("Concluído!")

Extraindo datas...
Calculando estatísticas mensais...
Calculando estatísticas anuais...
Concluído!


### AMOSTRA DO DATAFRAME TRATADO

In [48]:
df_estatisticas_mes.head()

,ano_distribuicao,mes_distribuicao,ano_baixa,mes_baixa,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
609394,2025,10,2025,10,ANICUNS,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",1,0,1,100.0
609269,2025,10,2025,10,APARECIDA DE GOIÂNIA,1ª Vara Criminal,1,0,1,100.0
609212,2025,10,2025,10,APARECIDA DE GOIÂNIA,1º Juizado Especial Cível,1,0,1,100.0
609160,2025,10,2025,10,APARECIDA DE GOIÂNIA,2ª Vara de Família e Sucessões,1,0,1,100.0
609290,2025,10,2025,10,APARECIDA DE GOIÂNIA,3ª Vara Criminal,1,0,1,100.0


In [49]:
df_estatisticas_anuais.head()

,ano_distribuicao,ano_baixa,comarca,serventia,Distribuídos,Baixados,Pendentes,Taxa de Congestionamento (%)
5235,2025,2025,ABADIÂNIA,Vara Judicial,1326,447,879,66.29
5561,2025,2025,ACREÚNA,"1ª Vara Judicial (Família e Sucessões, Infânci...",1218,426,792,65.02
5285,2025,2025,ACREÚNA,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",770,198,572,74.29
5169,2025,2025,ALEXÂNIA,Vara Judicial,2841,946,1895,66.70
5135,2025,2025,ALTO PARAÍSO DE GOIÁS,Vara Judicial,1375,302,1073,78.04


### SEPARAR CONJUNTOS DE TREINO (80%) E TESTE (20%) 

In [50]:
train, test_split = train_test_split(df_estatisticas_mes.copy(), test_size=0.2)

### AMOSTRA DO CONJUNTO TREINO

In [52]:
train.head()

,ano_distribuicao,mes_distribuicao,ano_baixa,mes_baixa,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
455401,2023,12,2025,6,ITABERAÍ,"1ª Vara Cível, Infância e da Juventude e Juiza...",114,3,31,91.18
24480,2022,1,2024,7,RIO VERDE,2ª Vara de Família e Sucessões,43,0,3,100.00
338508,2023,4,2024,8,GOIÂNIA,6ª Vara Criminal dos crimes punidos com reclus...,93,0,11,100.00
540137,2024,7,2025,3,GOIÂNIA,Auditoria Militar,67,5,22,81.48
526926,2024,6,2024,11,SÃO SIMÃO,Vara Judicial,167,4,47,92.16


### GRAVAR O CONJUNTO TREINO PRÉ-PROCESSADO

In [53]:
train.to_csv('datasets/train-processed.csv', index=False)

### AMOSTRA DO CONJUNTO TESTE

In [55]:
test_split.head()

,ano_distribuicao,mes_distribuicao,ano_baixa,mes_baixa,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
235297,2022,11,2025,6,SÃO LUÍS DE MONTES BELOS,"Vara Criminal (crime em geral, crimes dolosos ...",72,1,26,96.30
203206,2022,9,2023,1,ORIZONA,Vara Judicial,113,1,32,96.97
101342,2022,5,2022,5,ANÁPOLIS,1ª Vara Cível,107,1,37,97.37
560531,2024,10,2025,5,CALDAS NOVAS,12º CEJUSC REGIONAL VIRTUAL DO INTERIOR,30,0,2,100.00
293718,2023,2,2024,1,TRIBUNAL DE JUSTIÇA,GABINETE DESA NELMA BRANCO FERREIRA PERILO,97,1,2,66.67


### GRAVAR CONJUNTO TESTE PRÉ-PROCESSADO

In [56]:
test_split.to_csv('datasets/test_split.csv', index=False)